# Lecture 3 — Improving the Resume Scorer

In this notebook we systematically compare different prompt-engineering strategies for scoring resumes against a job description. Each strategy scores the same set of resumes and submits results to a shared leaderboard so we can see which techniques actually move the needle.

**Gold** resumes (prefixed `g`) are strong matches for the role. **Silver** resumes (prefixed `s`) are decent but weaker. **Wild** resumes (digit-prefixed) have unknown quality. A good scoring strategy should clearly separate gold from silver and wild.

In [ ]:
import os
import json
import random
import httpx
import pandas as pd
import numpy as np
from pydantic import BaseModel, Field
from resume_utils import load_resumes, load_job_requirements, structured_llm_call, submit_score

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
TEAM_NAME = "my-team"  # <-- Change this to your team name

if not OPENROUTER_API_KEY:
    raise RuntimeError(
        "Please set OPENROUTER_API_KEY before running this notebook.\n"
        "Get your key from: https://openrouter.ai/keys"
    )

print("Imports loaded")
print("API key configured")

## Load Data and Split into Tiers

We split resumes into three tiers based on their ID prefix:
- **Gold** (`g` prefix) — strong matches, should score highest
- **Silver** (`s` prefix) — decent matches, should score in the middle
- **Wild** (digit prefix) — unknown quality, used for calibration

In [ ]:
resumes = load_resumes('../data/resumes_final_with_gold_silver.csv')
job_req = load_job_requirements('../data/job_req_senior.md')

resume_list = sorted(resumes.values(), key=lambda r: r['ID'])
gold   = [r for r in resume_list if r['ID'].startswith('g')]
silver = [r for r in resume_list if r['ID'].startswith('s')]
wild   = [r for r in resume_list if r['ID'][0].isdigit()]

print(f"Gold resumes: {len(gold)} ({[r['ID'] for r in gold]})")
print(f"Silver resumes: {len(silver)} ({[r['ID'] for r in silver]})")
print(f"Wild resumes: {len(wild)}")

# We'll score all gold + silver + a sample of wild
random.seed(42)
wild_sample = random.sample(wild, min(10, len(wild)))
eval_resumes = gold + silver + wild_sample
print(f"\nEvaluation set: {len(eval_resumes)} resumes")

## Evaluation Helper

We measure each strategy with these metrics:
- **gold_mean / silver_mean / wild_mean** — average score per tier
- **gold_silver_gap** — how much higher gold scores vs silver (bigger is better)
- **rank_separation** — fraction of (gold, silver) pairs where gold > silver (1.0 = perfect separation)
- **gold_std / silver_std** — consistency within tiers (lower is better)

In [ ]:
def evaluate(scores: dict, gold_ids: set, silver_ids: set) -> dict:
    """
    Given a dict of {resume_id: score}, compute ordinal metrics.

    Returns dict with: gold_mean, silver_mean, wild_mean,
    gold_silver_gap, gold_wild_gap, rank_separation, gold_std, silver_std
    """
    gold_scores = [s for rid, s in scores.items() if rid in gold_ids]
    silver_scores = [s for rid, s in scores.items() if rid in silver_ids]
    wild_scores = [s for rid, s in scores.items() if rid not in gold_ids and rid not in silver_ids]

    gold_mean = np.mean(gold_scores) if gold_scores else 0
    silver_mean = np.mean(silver_scores) if silver_scores else 0
    wild_mean = np.mean(wild_scores) if wild_scores else 0

    # Rank separation: fraction of (g, s) pairs where gold > silver
    pairs = [(g, s) for g in gold_scores for s in silver_scores]
    rank_sep = sum(1 for g, s in pairs if g > s) / len(pairs) if pairs else 0

    return {
        'gold_mean': round(gold_mean, 2),
        'silver_mean': round(silver_mean, 2),
        'wild_mean': round(wild_mean, 2),
        'gold_silver_gap': round(gold_mean - silver_mean, 2),
        'gold_wild_gap': round(gold_mean - wild_mean, 2),
        'rank_separation': round(rank_sep, 3),
        'gold_std': round(np.std(gold_scores), 2) if gold_scores else 0,
        'silver_std': round(np.std(silver_scores), 2) if silver_scores else 0,
    }

gold_ids = {r['ID'] for r in gold}
silver_ids = {r['ID'] for r in silver}

## Submit Helper

`run_and_submit` scores every resume in a strategy, submits to the leaderboard, and prints evaluation metrics.

In [ ]:
def run_and_submit(strategy_name, scores, team=TEAM_NAME):
    """Submit all scores and return metrics."""
    for resume_id, score in scores.items():
        submit_score(team, resume_id, score)
    metrics = evaluate(scores, gold_ids, silver_ids)
    print(f"\n--- {strategy_name} ---")
    for k, v in metrics.items():
        print(f"  {k}: {v}")
    return {'strategy': strategy_name, **metrics}

results = []  # Accumulate results across strategies

## Strategy 1: Baseline

The monolithic prompt approach from Lecture 2. A single LLM call reads the resume and job requirements, then outputs a 0-100 score with reasoning. This is our control — every other strategy tries to beat it.

In [ ]:
baseline_schema = {
    "score": "integer 0-100",
    "reasoning": "string explaining the score"
}

baseline_prompt = """Score this resume against the job requirements on a 0-100 scale.
Consider technical skills match, years of experience, education, and overall fit.
Provide a score and brief reasoning."""

baseline_scores = {}
for i, resume in enumerate(eval_resumes):
    print(f"  Baseline {i+1}/{len(eval_resumes)}: {resume['ID']}")
    resp = structured_llm_call(
        api_key=OPENROUTER_API_KEY,
        prompt=baseline_prompt,
        context_data={"resume": resume['Resume_str'], "job_requirements": job_req},
        output_schema=baseline_schema,
        temperature=0.3,
    )
    if resp['result']:
        baseline_scores[resume['ID']] = resp['result'].get('score', 0)
    else:
        print(f"    ERROR: {resp['error']}")
        baseline_scores[resume['ID']] = 0

results.append(run_and_submit("baseline", baseline_scores))

## Strategy 2: Decomposition

Instead of asking the LLM to do everything at once, we break the task into focused extraction calls:
1. Extract years of experience
2. Extract technical skills (as a list)
3. Extract education level
4. Extract leadership experience

Then we combine the extractions with **deterministic Python scoring logic**. This gives us full control over how features map to a final score.

In [ ]:
# --- Extraction schemas ---
experience_schema = {
    "years_of_experience": "number — total years of relevant software development experience",
    "reasoning": "string — brief explanation of how you counted"
}

skills_schema = {
    "tech_skills": ["list of technology/framework names found in the resume"],
    "reasoning": "string — how you identified these"
}

education_schema = {
    "education_level": "string — one of: PhD, Masters, Bachelors, Associates, None",
    "field": "string — field of study",
    "reasoning": "string"
}

leadership_schema = {
    "has_leadership": "boolean — true if evidence of leading teams or mentoring",
    "reasoning": "string — cite evidence from the resume"
}

# --- Deterministic scoring function ---
TARGET_SKILLS = ["c#", ".net", "javascript", "react", "angular", "sql", "aws", "azure", "docker", "kubernetes"]

def compute_decomposed_score(years, skills, education_level, has_leadership):
    """Combine extracted features into a 0-100 score with explicit weights."""
    # Years of experience (max 25 pts): 5 pts per year up to 5 years, then diminishing
    years_score = min(25, years * 5) if years <= 5 else min(25, 25 + (years - 5))

    # Technical skills match (max 40 pts): 4 pts per matching target skill
    matched = [s for s in skills if any(t in s.lower() for t in TARGET_SKILLS)]
    skills_score = min(40, len(matched) * 4)

    # Education (max 20 pts)
    edu_map = {"phd": 20, "masters": 18, "bachelors": 15, "associates": 10, "none": 0}
    edu_score = edu_map.get(education_level.lower(), 10)

    # Leadership (max 15 pts)
    lead_score = 15 if has_leadership else 0

    return min(100, years_score + skills_score + edu_score + lead_score)

# --- Run decomposition on all eval resumes ---
decomp_scores = {}
for i, resume in enumerate(eval_resumes):
    print(f"  Decomp {i+1}/{len(eval_resumes)}: {resume['ID']}")
    ctx = {"resume": resume['Resume_str']}

    exp = structured_llm_call(OPENROUTER_API_KEY,
        "Extract the total years of software development experience from this resume.",
        ctx, experience_schema, temperature=0.1)

    sk = structured_llm_call(OPENROUTER_API_KEY,
        "List all technology skills, frameworks, and programming languages mentioned in this resume.",
        ctx, skills_schema, temperature=0.1)

    ed = structured_llm_call(OPENROUTER_API_KEY,
        "Extract the highest education level and field of study from this resume.",
        ctx, education_schema, temperature=0.1)

    ld = structured_llm_call(OPENROUTER_API_KEY,
        "Does this person have leadership or mentoring experience? Look for evidence of leading teams, mentoring juniors, or managing projects.",
        ctx, leadership_schema, temperature=0.1)

    years = exp['result'].get('years_of_experience', 0) if exp['result'] else 0
    skills = sk['result'].get('tech_skills', []) if sk['result'] else []
    edu = ed['result'].get('education_level', 'None') if ed['result'] else 'None'
    lead = ld['result'].get('has_leadership', False) if ld['result'] else False

    decomp_scores[resume['ID']] = compute_decomposed_score(years, skills, edu, lead)

results.append(run_and_submit("decomposition", decomp_scores))

## Strategy 3: Grounding with Citations

A common failure mode is the LLM "hallucinating" skills or experience. This strategy forces the model to **cite specific text from the resume** for every claim before scoring. If the model cannot find evidence, the claim should not contribute to the score.

In [ ]:
grounded_schema = {
    "claims": [
        {
            "category": "string — one of: experience, technical_skill, education, leadership",
            "claim": "string — what you are claiming about the candidate",
            "evidence": "string — exact quote from the resume supporting this claim",
            "confidence": "string — high, medium, or low"
        }
    ],
    "score": "integer 0-100 — based ONLY on high-confidence claims with direct evidence",
    "reasoning": "string — explain how evidence maps to the score"
}

grounded_prompt = """Score this resume against the job requirements on a 0-100 scale.

IMPORTANT: For every claim you make about the candidate, you MUST provide an exact
quote from the resume as evidence. If you cannot find a direct quote, mark confidence
as 'low' and do NOT let that claim influence your score.

Only high-confidence, evidence-backed claims should contribute to the final score."""

grounded_scores = {}
for i, resume in enumerate(eval_resumes):
    print(f"  Grounded {i+1}/{len(eval_resumes)}: {resume['ID']}")
    resp = structured_llm_call(
        api_key=OPENROUTER_API_KEY,
        prompt=grounded_prompt,
        context_data={"resume": resume['Resume_str'], "job_requirements": job_req},
        output_schema=grounded_schema,
        temperature=0.2,
    )
    if resp['result']:
        grounded_scores[resume['ID']] = resp['result'].get('score', 0)
    else:
        print(f"    ERROR: {resp['error']}")
        grounded_scores[resume['ID']] = 0

results.append(run_and_submit("grounded", grounded_scores))

## Strategy 4: Rubric + Few-Shot Anchoring

Instead of letting the LLM decide what matters, we define an **explicit weighted rubric** aligned to the job requirements. We also provide **anchor examples** so the model has concrete reference points for what different score levels look like.

| Criterion | Max Points |
|-----------|-----------|
| .NET / C# experience | 25 |
| JavaScript frameworks (React, Angular) | 20 |
| SQL / database skills | 15 |
| Cloud / DevOps (AWS, Azure, Docker) | 15 |
| Years of experience (5-10 target) | 15 |
| Soft skills / communication | 10 |

In [ ]:
rubric_schema = {
    "dotnet_csharp": "integer 0-25 — points for .NET and C# experience",
    "js_frameworks": "integer 0-20 — points for JavaScript framework experience (React, Angular, Vue)",
    "sql_databases": "integer 0-15 — points for SQL and database skills",
    "cloud_devops": "integer 0-15 — points for cloud platforms and DevOps (AWS, Azure, Docker, K8s)",
    "years_experience": "integer 0-15 — points for years of experience (0-2yrs: 3pts, 3-4yrs: 7pts, 5-7yrs: 12pts, 8+yrs: 15pts)",
    "soft_skills": "integer 0-10 — points for communication, teamwork, mentoring",
    "total_score": "integer 0-100 — sum of all category scores",
    "reasoning": "string — brief justification per category"
}

rubric_prompt = """Score this resume against the job requirements using the EXACT rubric below.

RUBRIC (total 100 points):
1. .NET / C# experience (0-25 pts): Award based on depth and recency of C#/.NET work.
2. JavaScript frameworks (0-20 pts): React, Angular, or Vue experience.
3. SQL / databases (0-15 pts): SQL Server, PostgreSQL, query optimization, schema design.
4. Cloud / DevOps (0-15 pts): AWS, Azure, Docker, Kubernetes, CI/CD pipelines.
5. Years of experience (0-15 pts): 0-2yrs=3pts, 3-4yrs=7pts, 5-7yrs=12pts, 8+=15pts.
6. Soft skills (0-10 pts): Communication, mentoring, cross-team collaboration.

ANCHOR EXAMPLES:
- A candidate with 8 years of C#/.NET, React, SQL Server, AWS, and team lead experience = ~85-95
- A candidate with 3 years of Python/Django, some SQL, no cloud experience = ~15-25
- A candidate with 5 years of C#, no JS frameworks, basic SQL, no cloud = ~40-55

Score each category independently. The total_score MUST equal the sum of all categories."""

rubric_scores = {}
for i, resume in enumerate(eval_resumes):
    print(f"  Rubric {i+1}/{len(eval_resumes)}: {resume['ID']}")
    resp = structured_llm_call(
        api_key=OPENROUTER_API_KEY,
        prompt=rubric_prompt,
        context_data={"resume": resume['Resume_str'], "job_requirements": job_req},
        output_schema=rubric_schema,
        temperature=0.2,
    )
    if resp['result']:
        rubric_scores[resume['ID']] = resp['result'].get('total_score', 0)
    else:
        print(f"    ERROR: {resp['error']}")
        rubric_scores[resume['ID']] = 0

results.append(run_and_submit("rubric", rubric_scores))

## Strategy 5: Chain-of-Thought via Schema Ordering

A subtle but powerful trick: by placing the `reasoning` field **before** the `score` field in the output schema, we force the model to write out its reasoning first, then commit to a number. This is a form of chain-of-thought prompting that works through the structure of the output itself.

Compare the schema ordering:
- Baseline: `{"score": ..., "reasoning": ...}` -- model picks a number, then justifies it
- CoT: `{"reasoning": ..., "score": ...}` -- model reasons first, then scores

In [ ]:
# NOTE: reasoning comes BEFORE score — this forces the model to think first
cot_schema = {
    "reasoning": "string — step-by-step analysis of the resume against each job requirement. Be thorough.",
    "strengths": ["list of specific strengths relevant to the job"],
    "weaknesses": ["list of specific gaps or weaknesses relative to the job"],
    "score": "integer 0-100 — final score based on the reasoning above"
}

cot_prompt = """Score this resume against the job requirements on a 0-100 scale.

IMPORTANT: Think step by step. First analyze the resume thoroughly — identify strengths,
weaknesses, and how well the candidate matches each requirement. Write out your full
reasoning BEFORE deciding on a score. Your score must follow logically from your analysis."""

cot_scores = {}
for i, resume in enumerate(eval_resumes):
    print(f"  CoT {i+1}/{len(eval_resumes)}: {resume['ID']}")
    resp = structured_llm_call(
        api_key=OPENROUTER_API_KEY,
        prompt=cot_prompt,
        context_data={"resume": resume['Resume_str'], "job_requirements": job_req},
        output_schema=cot_schema,
        temperature=0.2,
    )
    if resp['result']:
        cot_scores[resume['ID']] = resp['result'].get('score', 0)
    else:
        print(f"    ERROR: {resp['error']}")
        cot_scores[resume['ID']] = 0

results.append(run_and_submit("cot", cot_scores))

## Strategy 6: Student Choice Extension

Pick one (or combine several) of these advanced techniques:

- **Self-consistency**: Run the same prompt 3-5 times, take the median score. Reduces variance.
- **LLM-as-judge**: Have one LLM call score the resume, then a second call critique and adjust the score.
- **Cascade / filter**: Use a cheap fast call to classify resumes as "strong / maybe / weak", then only run the expensive detailed scoring on "strong" and "maybe" candidates.
- **Ensemble**: Combine scores from multiple strategies above (e.g., average of rubric + CoT).
- **Your own idea**: Any other technique you think will improve separation.

Fill in the skeleton below with your approach.

In [ ]:
# --- YOUR CUSTOM STRATEGY ---
# Replace the placeholder below with your approach.
# The only requirement: produce a dict of {resume_id: score} and submit it.

custom_schema = {
    # TODO: Define your output schema
    "score": "integer 0-100",
    "reasoning": "string"
}

custom_prompt = """
TODO: Write your custom prompt here.
"""

custom_scores = {}
for i, resume in enumerate(eval_resumes):
    print(f"  Custom {i+1}/{len(eval_resumes)}: {resume['ID']}")

    # TODO: Implement your strategy here
    # Examples:
    #   - Self-consistency: call structured_llm_call 3x, take median
    #   - LLM-as-judge: first call scores, second call critiques
    #   - Ensemble: average rubric_scores and cot_scores

    resp = structured_llm_call(
        api_key=OPENROUTER_API_KEY,
        prompt=custom_prompt,
        context_data={"resume": resume['Resume_str'], "job_requirements": job_req},
        output_schema=custom_schema,
        temperature=0.3,
    )
    if resp['result']:
        custom_scores[resume['ID']] = resp['result'].get('score', 0)
    else:
        print(f"    ERROR: {resp['error']}")
        custom_scores[resume['ID']] = 0

results.append(run_and_submit("custom", custom_scores))

## Final Comparison

Compare all strategies side-by-side. The key metrics to watch:
- **gold_silver_gap**: Higher is better (your strategy separates good from mediocre candidates)
- **rank_separation**: Closer to 1.0 is better (every gold resume scores above every silver resume)
- **gold_std / silver_std**: Lower is better (consistent scoring within tiers)

In [ ]:
import matplotlib.pyplot as plt

# Build comparison DataFrame
comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))

# Bar chart of gold_silver_gap per strategy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gold-Silver gap
axes[0].bar(comparison_df['strategy'], comparison_df['gold_silver_gap'], color='steelblue')
axes[0].set_title('Gold-Silver Gap by Strategy')
axes[0].set_ylabel('Score Gap (higher is better)')
axes[0].tick_params(axis='x', rotation=45)

# Rank separation
axes[1].bar(comparison_df['strategy'], comparison_df['rank_separation'], color='coral')
axes[1].set_title('Rank Separation by Strategy')
axes[1].set_ylabel('Fraction (closer to 1.0 is better)')
axes[1].set_ylim(0, 1.05)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Fetch metrics from the leaderboard API to compare with other teams
try:
    with httpx.Client(timeout=10) as client:
        resp = client.get(
            "http://ai-leaderboard.site/lecture3/api/metrics",
            headers={"X-API-Key": "lecture3-secret-key"},
        )
        resp.raise_for_status()
        all_metrics = resp.json()
    leaderboard_df = pd.DataFrame(all_metrics)
    print("=== Leaderboard Metrics (All Teams) ===")
    print(leaderboard_df.sort_values('gold_silver_gap', ascending=False).to_string(index=False))
except Exception as e:
    print(f"Could not fetch leaderboard metrics: {e}")